In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pathlib
import copy
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 1. THIẾT LẬP THAM SỐ CỐT LÕI
# ==========================================
GroupID = "04" 
IMAGE_SIZE = 224 # KHÔNG ĐƯỢC ĐỔI [cite: 472-476]
BATCH_SIZE = 32
EPOCHS = 120
LEARNING_RATE = 1e-3

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. XỬ LÝ DỮ LIỆU (DATA PIPELINE)
# ==========================================
data_dir = pathlib.Path("/kaggle/input/datasets/huynhthethien/radarcommunsignaldata2026train")

# Chỉ giữ các Transform cơ bản vì ta sẽ dùng Mixup Augmentation bên dưới
base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

base_dataset = datasets.ImageFolder(root=str(data_dir), transform=base_transform)
num_classes = len(base_dataset.classes)

train_size = int(0.8 * len(base_dataset))
val_size = len(base_dataset) - train_size

generator = torch.Generator().manual_seed(42) # Cố định seed
train_dataset, val_dataset = random_split(
    base_dataset, [train_size, val_size], generator=generator
)

# Tách biệt đối tượng Transform để an toàn
dataset_val = copy.deepcopy(base_dataset)
dataset_val.transform = base_transform
val_dataset.dataset = dataset_val

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 3. KIẾN TRÚC MÔ HÌNH (~98.500 THAM SỐ)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class EfficientBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, 
                                   stride=stride, padding=1, groups=in_channels, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.depthwise(x)))
        out = self.bn2(self.pointwise(out))
        out = self.se(out)
        out += residual
        return self.relu(out)

class BasicCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Mở rộng Kênh và Kernel Size để bắt được nhiều thông tin hơn ngay từ đầu
        self.init_conv = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True)
        )
        
        # Thiết kế mạng tiệm cận 100k params: 24 -> 48 -> 80 -> 128 -> 144
        self.blocks = nn.Sequential(
            EfficientBlock(24, 48, stride=2),   
            EfficientBlock(48, 80, stride=2),   
            EfficientBlock(80, 128, stride=2),  
            EfficientBlock(128, 144, stride=2)  
        )
        
        self.adapt_pool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(144, num_classes)
        
    def forward(self, x):
        x = self.init_conv(x)
        x = self.blocks(x)
        x = self.adapt_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x) # Trả về raw logits [cite: 501-502]

model = BasicCNN(num_classes).to(DEVICE)

# Làm mịn nhãn để chống Overfitting
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

# Sốc nhiệt Learning Rate: Khởi động lại mỗi 15 epoch
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=1)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*60)
print("THÔNG TIN CẤU HÌNH VÀ MÔ HÌNH")
print("="*60)
print(f"Device:              {DEVICE}")
print(f"Tổng số tham số:     {total_params} (Yêu cầu đồ án: < 100,000)")
print("Kỹ thuật áp dụng:    Mixup, Label Smoothing, Sốc Nhiệt LR, SE-Block")
print("="*60)

# ==========================================
# 4. HÀM MIXUP AUGMENTATION & VẼ ĐỒ THỊ
# ==========================================
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def plot_and_save_metrics(history, class_names, all_labels, all_preds, group_id):
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, history['train_loss'], label='Training Loss', marker='o', markersize=4)
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss', marker='s', markersize=4)
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(f'{group_id}_Loss_Chart.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, history['train_acc'], label='Training Accuracy', marker='o', markersize=4)
    plt.plot(epochs_range, history['val_acc'], label='Validation Accuracy', marker='s', markersize=4)
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(f'{group_id}_Accuracy_Chart.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix on Validation Set')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig(f'{group_id}_Confusion_Matrix.png', dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 5. QUÁ TRÌNH HUẤN LUYỆN CHÍNH
# ==========================================
def train_model(model, train_loader, val_loader):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Áp dụng Mixup với xác suất 50% để giữ sự cân bằng
            if np.random.rand() > 0.5:
                images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.2)
                optimizer.zero_grad()
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            else:
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total
        
        # Cập nhật Scheduler
        scheduler.step()

        # Đánh giá trên tập Validation
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct / total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f} --> ĐÃ LƯU KỶ LỤC MỚI!")
        else:
            print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}")
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

    print(f"\nHuấn luyện hoàn tất! Validation Accuracy cao nhất: {best_val_acc:.4f}")
    
    # Phục hồi trọng số tốt nhất trước khi test cuối cùng
    model.load_state_dict(best_model_wts)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print("\n--- Báo cáo Đánh giá (Classification Report trên mô hình tốt nhất) ---")
    print(classification_report(all_labels, all_preds, target_names=base_dataset.classes))
    plot_and_save_metrics(history, base_dataset.classes, all_labels, all_preds, GroupID)

train_model(model, train_loader, val_loader)

# ==========================================
# 6. XUẤT MÔ HÌNH (KHÔNG ĐƯỢC CHỈNH SỬA)
# ==========================================
############################################
# DO NOT MODIFY THIS SECTION
############################################
model.eval()
example_input = torch.randn(1,3, IMAGE_SIZE, IMAGE_SIZE).to (DEVICE)
traced_model = torch.jit.trace(model, example_input)

model_name = f"{GroupID}_DeepLearningProject_TrainedModel.pt"
traced_model.save(model_name)
print("\nModel saved:", model_name)

THÔNG TIN CẤU HÌNH VÀ MÔ HÌNH
Device:              cuda:0
Tổng số tham số:     98508 (Yêu cầu đồ án: < 100,000)
Kỹ thuật áp dụng:    Mixup, Label Smoothing, Sốc Nhiệt LR, SE-Block
Epoch 01/120 | Train Loss: 1.4489 - Acc: 0.4961 | Val Loss: 1.0536 - Acc: 0.7546 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 02/120 | Train Loss: 1.1731 - Acc: 0.6075 | Val Loss: 0.9578 - Acc: 0.7936 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 03/120 | Train Loss: 1.0841 - Acc: 0.6434 | Val Loss: 0.8885 - Acc: 0.8276 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 04/120 | Train Loss: 1.0540 - Acc: 0.6511 | Val Loss: 0.8512 - Acc: 0.8536 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 05/120 | Train Loss: 1.0255 - Acc: 0.6620 | Val Loss: 0.8394 - Acc: 0.8583 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 06/120 | Train Loss: 0.9987 - Acc: 0.6727 | Val Loss: 0.8179 - Acc: 0.8633 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 07/120 | Train Loss: 1.0072 - Acc: 0.6604 | Val Loss: 0.8180 - Acc: 0.8633
Epoch 08/120 | Train Loss: 0.9768 - Acc: 0.6813 | Val Loss: 0.8149 - Acc: 0.8650 --> ĐÃ LƯU KỶ LỤC MỚI!
Epoch 09/12